# Fine-tuning BART-base

Switch from `google/flan-t5-small` to `facebook/bart-base` because the FLAN-T5 SentencePiece vocabulary cannot represent JSON braces (`{`, `}`) or several Unicode units (`±`, `µ`, `×`, `≤`, `≥`). BART's byte-level BPE round-trips these characters without `<unk>`.

Reference: [facebook/bart-base model card](https://huggingface.co/facebook/bart-base).


## 1. Install dependencies

In [16]:
%pip install -q "transformers>=4.46,<5" "torch>=2.6" datasets accelerate

## 2. Set up the runtime


In [17]:
import platform
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed

MODEL_NAME = "facebook/bart-base"
SEED = 42
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.16
PyTorch: 2.14.1+cu130
Transformers: 4.57.6
Device: cuda
GPU: Tesla T4


## 3. Load the untuned model



In [18]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    use_safetensors=True,
)
model = model.to(device)
model.eval()

assert model.config.is_encoder_decoder
assert not model.training
print("Loaded:", MODEL_NAME)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Model device:", next(model.parameters()).device)
print("Tokenizer limit:", tokenizer.model_max_length)
print("Decoder start token id:", model.config.decoder_start_token_id)


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loaded: facebook/bart-base
Parameters: 139,420,416
Model device: cuda:0
Tokenizer limit: 1000000000000000019884624838656
Decoder start token id: 2


## 4. Load and Prepare dataset
Data is split by device family under `data/splits/`.
Just have to convert each record to input-target pair:
* Input: the `passage` text

* Target: requirements list serialised as a string

In [19]:
!git clone https://github.com/vet3whale/nlp_project.git
%cd nlp_project

Cloning into 'nlp_project'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 71 (delta 17), reused 71 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 14.44 MiB | 10.42 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/nlp_project/nlp_project


load the three splits

In [20]:
import json, os
import pandas as pd

SPLIT_DIR = "data/splits"

def load_split(name):
    for ext in (".json", ".jsonl"):
        path = os.path.join(SPLIT_DIR, f"{name}{ext}")
        if os.path.exists(path):
            with open(path, encoding="utf-8") as f:
                if ext == ".json":
                    return json.load(f)
                return [json.loads(line) for line in f if line.strip()]
    raise FileNotFoundError(f"No split found for {name} in {SPLIT_DIR}")

train_records = load_split("train")
val_records   = load_split("validation")
test_records  = load_split("test")

verify data before building

In [21]:
# The splits do not carry a 'family' key. Recover the family from the
# per-device files in data/extracted/, joining on passage text.
import glob

family_by_passage = {}
for path in glob.glob("data/extracted/*.json"):
    fam = os.path.splitext(os.path.basename(path))[0]
    with open(path, encoding="utf-8") as f:
        for rec in json.load(f):
            family_by_passage[rec["passage"].strip()] = fam

def family_of(r):
    return family_by_passage.get(r["passage"].strip(), "UNKNOWN")

for name, recs in [("train", train_records), ("val", val_records), ("test", test_records)]:
    fams = sorted({family_of(r) for r in recs})
    negs = sum(1 for r in recs if not (r.get("requirements") or []))
    print(f"{name}: {len(recs)} examples | families={fams} | {negs} negatives")

train_f = {family_of(r) for r in train_records}
val_f   = {family_of(r) for r in val_records}
test_f  = {family_of(r) for r in test_records}
print("overlaps:", train_f & val_f, train_f & test_f, val_f & test_f)
assert not (train_f & val_f) and not (train_f & test_f) and not (val_f & test_f)

train: 135 examples | families=['hdc1080', 'ina219', 'opt3001', 'sgp30', 'tmag5273', 'tmp117', 'veml7700'] | 26 negatives
val: 40 examples | families=['mcp9808', 'mpl3115a2'] | 8 negatives
test: 45 examples | families=['bme280', 'fdc1004', 'sht31'] | 9 negatives
overlaps: set() set() set()


build input/target pairs

In [22]:
INSTRUCTION = ("Extract numerical operating requirements and their conditions "
               "from the datasheet passage. Return JSON.")

def serialize_requirements(reqs):
    return json.dumps(reqs or [], ensure_ascii=False, separators=(",", ":"))

def build_frame(records):
    rows = []
    for r in records:
        reqs = r.get("requirements") or []
        rows.append({
            "input": f"{INSTRUCTION}\n\n{r['passage'].strip()}",
            "target": serialize_requirements(reqs),
            "family": family_of(r),
            "n_reqs": len(reqs),
        })
    return pd.DataFrame(rows)

df_train = build_frame(train_records)
df_val   = build_frame(val_records)
df_test  = build_frame(test_records) # built for later; not used in training

## 5. Tokenise inputs and targets
BART uses byte-level BPE, so all ASCII punctuation and Unicode units round-trip. Measure lengths first, then choose a target cap from the observed maximum.


In [23]:
import numpy as np

def raw_lengths(texts, is_target=False):
    kw = {"text_target": texts} if is_target else {"text": texts}
    enc = tokenizer(**kw, truncation=False, padding=False, add_special_tokens=True)
    return [len(ids) for ids in enc["input_ids"]]

for name, df in [("train", df_train), ("val", df_val)]:
    li = raw_lengths(df["input"].tolist())
    lt = raw_lengths(df["target"].tolist(), is_target=True)
    print(f"{name} input  tokens: p50={np.percentile(li,50):.0f} p95={np.percentile(li,95):.0f} max={max(li)}")
    print(f"{name} target tokens: p50={np.percentile(lt,50):.0f} p95={np.percentile(lt,95):.0f} max={max(lt)}")

train input  tokens: p50=70 p95=112 max=151
train target tokens: p50=70 p95=153 max=257
val input  tokens: p50=72 p95=90 max=118
val target tokens: p50=60 p95=116 max=137


tokenise with correct cap

In [24]:
MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 384

def tokenize(batch):
    enc = tokenizer(
        batch["input"],
        max_length=MAX_INPUT_LEN, truncation=True, padding=False,
        add_special_tokens=True,
    )
    lab = tokenizer(
        text_target=batch["target"],
        max_length=MAX_TARGET_LEN, truncation=True, padding=False,
    )
    enc["labels"] = lab["input_ids"]
    return enc

In [25]:
def decode_labels(ids):
    return tokenizer.decode([t for t in ids if t != -100], skip_special_tokens=True)

bad = []
for df, name in [(df_train, "train"), (df_val, "val")]:
    for i, row in df.iterrows():
        enc = tokenizer(text_target=row["target"], truncation=False, add_special_tokens=True)
        dec = decode_labels(enc["input_ids"])
        if dec != row["target"]:
            bad.append((name, i))
print("round-trip mismatches:", len(bad))
assert len(bad) == 0, f"{len(bad)} mismatches — do not train yet"

round-trip mismatches: 0


check unicode characters are correctly interpreted

In [26]:
for ch in ['{', '}', '<', '>', '<=', '>=', '°', '±', 'µ', '×', '≤', '≥', '—', ';']:
    ids = tokenizer(ch, add_special_tokens=False)["input_ids"]
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f"{ch!r:6} -> {toks}  unk={'<unk>' in toks}")


'{'    -> ['{']  unk=False
'}'    -> ['}']  unk=False
'<'    -> ['<']  unk=False
'>'    -> ['>']  unk=False
'<='   -> ['<', '=']  unk=False
'>='   -> ['>', '=']  unk=False
'°'    -> ['Â°']  unk=False
'±'    -> ['Â±']  unk=False
'µ'    -> ['Â', 'µ']  unk=False
'×'    -> ['ÃĹ']  unk=False
'≤'    -> ['âī', '¤']  unk=False
'≥'    -> ['âī', '¥']  unk=False
'—'    -> ['âĢĶ']  unk=False
';'    -> [';']  unk=False


In [27]:
EOS_ID = tokenizer.eos_token_id
lab_ids = tokenizer(text_target=df_train["target"].tolist(),
                    truncation=False, padding=False)["input_ids"]
print("eos:", EOS_ID, repr(tokenizer.eos_token))
print("all labels end with EOS:", all(s[-1] == EOS_ID for s in lab_ids))
print("labels over cap:", sum(1 for s in lab_ids if len(s) > MAX_TARGET_LEN))
assert all(s[-1] == EOS_ID for s in lab_ids)
assert sum(1 for s in lab_ids if len(s) > MAX_TARGET_LEN) == 0

eos: 2 '</s>'
all labels end with EOS: True
labels over cap: 0


In [31]:
from datasets import Dataset

keep = ["input", "target"]
ds_train = Dataset.from_pandas(df_train[keep], preserve_index=False)
ds_val   = Dataset.from_pandas(df_val[keep],   preserve_index=False)
ds_test  = Dataset.from_pandas(df_test[keep],  preserve_index=False)

tok_train = ds_train.map(tokenize, batched=True, remove_columns=ds_train.column_names)
tok_val   = ds_val.map(tokenize,   batched=True, remove_columns=ds_val.column_names)
tok_test  = ds_test.map(tokenize,  batched=True, remove_columns=ds_test.column_names)

for n, d in [("train", tok_train), ("val", tok_val), ("test", tok_test)]:
    print(n, len(d), "| cols:", d.column_names,
          "| max label len:", max(len(x) for x in d["labels"]))

Map:   0%|          | 0/135 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

train 135 | cols: ['input_ids', 'attention_mask', 'labels'] | max label len: 257
val 40 | cols: ['input_ids', 'attention_mask', 'labels'] | max label len: 137
test 45 | cols: ['input_ids', 'attention_mask', 'labels'] | max label len: 113


## 6. Data collator
Collator sits between tokenised dataset and model's forward pass.
Does 2 things:
1. Pads each batch to longest sequence.  
    The collator pads each batch to the longest member of that batch, not to the global `MAX_INPUT_LEN/MAX_TARGET_LEN`. Short batches stay cheap.
2. Masks label padding with -100.  
    PyTorch cross-entropy uses ignore_index=-100, so padding positions in labels are excluded from loss. Without this, the model would learn to predict padding tokens.

In [32]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

sample_batch = data_collator([tok_train[i] for i in range(4)])
print({k: tuple(v.shape) for k, v in sample_batch.items()})
print("padding token in labels:", (sample_batch["labels"] == -100).any().item())

{'input_ids': (4, 79), 'attention_mask': (4, 79), 'labels': (4, 77), 'decoder_input_ids': (4, 77)}
padding token in labels: True


## 7. Training arguments

In [33]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-base-datasheet",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
    report_to="none",
)

## 8. Trainer and training

In [34]:
# build trainer
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=data_collator,
)

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.853000,0.426309
2,0.589500,0.208212
3,0.218800,0.170805
4,0.198200,0.152204
5,0.104100,0.159245
6,0.101600,0.146196
7,0.058800,0.148963
8,0.071900,0.143484
9,0.058800,0.160628
10,0.053400,0.148246


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=340, training_loss=0.18667820619747919, metrics={'train_runtime': 559.036, 'train_samples_per_second': 4.83, 'train_steps_per_second': 0.608, 'total_flos': 181138172866560.0, 'train_loss': 0.18667820619747919, 'epoch': 20.0})

## 9. Save the fine-tuned model

In [36]:
trainer.save_model("./bart-base-datasheet-finetuned")
tokenizer.save_pretrained("./bart-base-datasheet-finetuned")

import json
config = {
    "model": MODEL_NAME,
    "instruction": INSTRUCTION,
    "max_input_len": MAX_INPUT_LEN,
    "max_target_len": MAX_TARGET_LEN,
    "seed": SEED,
}
json.dump(config, open("./bart-base-datasheet-finetuned/training_config.json", "w"),
          ensure_ascii=False, indent=2)
print("saved")

saved


10. Evaluate Model

In [43]:
import json, numpy as np

def safe_decode(ids):
    arr = np.asarray(ids)
    arr = np.where(arr < 0, tokenizer.pad_token_id, arr)
    return tokenizer.batch_decode(arr, skip_special_tokens=True)

def parse(s):
    try:
        return json.loads(s)
    except Exception:
        return None

model.eval()
preds = trainer.predict(
    tok_val,
    max_new_tokens=MAX_TARGET_LEN,
    num_beams=8,
    length_penalty=2.0,
)

decoded_preds  = safe_decode(preds.predictions)
gold_targets   = df_val["target"].tolist()

# --- metrics ---
exact = 0
tp = fp = fn = 0
neg_total = neg_correct = 0

for p_str, g_str in zip(decoded_preds, gold_targets):
    p, g = parse(p_str), parse(g_str)
    if p is not None and g is not None and p == g:
        exact += 1
    if g is None:
        continue
    if len(g) == 0:
        neg_total += 1
        if p == []:
            neg_correct += 1
        continue
    if p is None:
        fn += len(g)
        continue
    used = [False] * len(g)
    for pr in p:
        hit = next((i for i, gr in enumerate(g) if not used[i] and pr == gr), None)
        if hit is None:
            fp += 1
        else:
            used[hit] = True
            tp += 1
    fn += sum(1 for u in used if not u)

prec = tp / (tp + fp) if (tp + fp) else 0.0
rec  = tp / (tp + fn) if (tp + fn) else 0.0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

print(f"val exact match : {exact}/{len(decoded_preds)} ({exact/len(decoded_preds):.3f})")
print(f"req-level P/R/F1: {prec:.3f} / {rec:.3f} / {f1:.3f}   (TP={tp} FP={fp} FN={fn})")
print(f"negatives       : {neg_correct}/{neg_total} correct")
print(f"val loss        : {preds.metrics.get('test_loss', 'n/a')}")

# --- side-by-side: show 5 mismatched positives + 2 negatives ---
mismatch_idx = [i for i in range(len(decoded_preds))
                if parse(decoded_preds[i]) != parse(gold_targets[i])]
print(f"\n=== showing up to 5 mismatches ===")
for i in mismatch_idx[:5]:
    print(f"\n--- val[{i}] ---")
    print("PASSAGE:", df_val["input"].iloc[i][-250:].replace("\n", " ")[:250])
    print("PRED  :", decoded_preds[i][:400])
    print("GOLD  :", gold_targets[i][:400])

neg_idx = [i for i, g in enumerate(gold_targets) if parse(g) == []][:2]
print(f"\n=== showing 2 negatives ===")
for i in neg_idx:
    print(f"\n--- val[{i}] (negative) ---")
    print("PASSAGE:", df_val["input"].iloc[i][-250:].replace("\n", " ")[:250])
    print("PRED  :", decoded_preds[i][:200])
    print("GOLD  :", gold_targets[i][:200])

val exact match : 13/40 (0.325)
req-level P/R/F1: 1.000 / 0.447 / 0.618   (TP=21 FP=0 FN=26)
negatives       : 4/8 correct
val loss        : 0.1434837430715561

=== showing up to 5 mismatches ===

--- val[2] ---
PASSAGE: Extract numerical operating requirements and their conditions from the datasheet passage. Return JSON.  9.3 Mechanical characteristics VDD = 2.5 V; T = 25 °C; Calibrated range Pressure measurement range MIN: 50 | MAX: 110 | UNIT: kPa
PRED  : [{"parameter":"Pressure measurement range","operator":">=","value":50,"unit":"kPa","condition":"VDD = 2.5 V; T = 25 °C; Calibrated range"}]
GOLD  : [{"parameter":"Pressure measurement range","operator":">=","value":50,"unit":"kPa","condition":"VDD = 2.5 V; T = 25 °C; Calibrated range"},{"parameter":"Pressure measurement range","operator":"<=","value":110,"unit":"kPa","condition":"VDD = 2.5 V; T = 25 °C; Calibrated range"}]

--- val[3] ---
PASSAGE: Extract numerical operating requirements and their conditions from the datasheet pass